# Plan Log Analysis

Analyze **`TF_LOG=json`** trace output captured during `terraform plan` (for example via `TF_LOG_PATH`).

Parses graph refresh lines, planned changes, and `PlanResourceChange` RPC activity from provider/core diagnostic JSON.

Capture example:

```bash
export TF_LOG=json
export TF_LOG_PATH=import-queue-plan-tf_json.log
terraform plan
export TERRAFORM_LOG_PATH=import-queue-plan-tf_json.log
```

For **`terraform plan -json`** UI output, use `plan-output-analysis.ipynb` instead. Run `whatisit.ipynb` if unsure.

In [ ]:
import pandas as pd
import commonlib.prep_plan_log_data as prep_plan_log_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as cfg

In [ ]:
c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)
parsed_records = prep_plan_log_data.read_json_from_file(c.TERRAFORM_LOG_PATH)
normalized_records = prep_plan_log_data.normalize_records(parsed_records)
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No TF_LOG plan trace records found. Capture with TF_LOG=json during plan "
        "or use plan-output-analysis.ipynb for terraform plan -json UI output."
    )

print(sorted(df["type"].drop_duplicates().tolist()))

In [ ]:
starts = (
    df[df["type"] == "refresh_start"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
starts["refresh_start_timestamp"] = pd.to_datetime(starts["timestamp"], utc=True, errors="coerce")
starts["run"] = starts.groupby("resource_id").cumcount() + 1
starts = starts[
    ["resource_id", "run", "refresh_start_timestamp", "resource", "resource_type", "resource_name"]
]

ends = (
    df[df["type"] == "refresh_complete"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
ends["refresh_complete_timestamp"] = pd.to_datetime(ends["timestamp"], utc=True, errors="coerce")
ends["run"] = ends.groupby("resource_id").cumcount() + 1
ends = ends[["resource_id", "run", "refresh_complete_timestamp"]]

df_merged_refresh = starts.merge(ends, on=["resource_id", "run"], how="inner")
df_merged_refresh["time_diff_minutes"] = (
    (df_merged_refresh["refresh_complete_timestamp"] - df_merged_refresh["refresh_start_timestamp"])
    .dt.total_seconds()
    / 60
)

df_planned_change = df[df["type"] == "planned_change"]

## Type Analysis

In [ ]:
gencharts.generate_plt_by_resource_type(df, "refresh_start", top_n=10)
gencharts.generate_plt_by_resource_type(df, "refresh_complete", top_n=10)
gencharts.generate_plt_by_resource_type(df, "planned_change", top_n=10)

## Duration Analysis

In [ ]:
if not df_merged_refresh.empty:
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="total", top_n=10)
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="average", top_n=10)
else:
    print("No matched refresh_start/refresh_complete pairs to chart.")

## Longest Refresh Times

In [ ]:
df_merged_refresh[
    ["resource", "resource_type", "refresh_start_timestamp", "refresh_complete_timestamp", "time_diff_minutes", "run"]
].copy().sort_values(by="time_diff_minutes", ascending=False).head(20)